# Nonprofit Risk Model - EDA & Analysis

Exploring the IRS BMF dataset merged with revocation records. Looking at feature distributions, class balance, and what makes a nonprofit high-risk.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_DIR = Path("../data/processed")

In [2]:
X = pd.read_parquet(DATA_DIR / "features.parquet")
y = pd.read_parquet(DATA_DIR / "labels.parquet").squeeze()

print(f"Dataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"\nFeatures:\n {list(X.columns)}")

Dataset shape: (1,821,403, 11)
Labels shape: (1,821,403,)

Features:
 ['asset_code_usd', 'income_code_usd', 'revenue_amount', 'subsection_code', 'foundation_code', 'ntee_major', 'years_since_ruling', 'years_since_filing', 'filing_req_code', 'deductibility_code', 'state']


## Class Distribution

The dataset is heavily imbalanced - most nonprofits are in good standing.

In [3]:
counts = y.value_counts()
print(f"Active (0):  {counts[0]:,} ({counts[0]/len(y)*100:.2f}%)")
print(f"Revoked (1):    {counts[1]:,} ({counts[1]/len(y)*100:.2f}%)")
print(f"\nImbalance ratio: {counts[0]/counts[1]:.1f}:1")

fig, ax = plt.subplots()
ax.bar(["Active", "Revoked"], [counts[0], counts[1]], color=["#4CAF50", "#F44336"])
ax.set_ylabel("Count")
ax.set_title("Class Distribution")
for i, v in enumerate([counts[0], counts[1]]):
    ax.text(i, v + 10000, f"{v:,}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

Active (0):  1,766,961 (97.01%)
Revoked (1):    54,442 (2.99%)

Imbalance ratio: 32.5:1


<Figure size 1000x600 with 1 Axes>

32:1 imbalance is significant. We'll need `scale_pos_weight` in XGBoost and PR-AUC as the main metric (ROC-AUC can be misleading with this kind of skew).

## Numeric Feature Distributions

In [4]:
numeric_cols = ["asset_code_usd", "income_code_usd", "revenue_amount",
                "years_since_ruling", "years_since_filing"]
X[numeric_cols].describe().T[["count", "mean", "std", "min", "50%", "max"]]

,count,mean,std,min,50%,max
asset_code_usd,1821403,"412,891","3,241,567",0,"25,000","75,000,000"
income_code_usd,1821403,"287,443","2,104,892",0,"25,000","75,000,000"
revenue_amount,1821403,"1,045,231","48,291,034",0,"42,187","31,849,201,000"
years_since_ruling,1821403,24.3,18.7,0,21,85
years_since_filing,1821403,2.1,4.8,0,1,30


In [5]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Years since ruling by class
for label, color in [(0, "#4CAF50"), (1, "#F44336")]:
    mask = y == label
    axes[0, 0].hist(X.loc[mask, "years_since_ruling"].dropna(), bins=40,
                    alpha=0.6, color=color, label="Revoked" if label else "Active", density=True)
axes[0, 0].set_title("Years Since Ruling")
axes[0, 0].legend()

# Years since filing by class
for label, color in [(0, "#4CAF50"), (1, "#F44336")]:
    mask = y == label
    axes[0, 1].hist(X.loc[mask, "years_since_filing"].dropna(), bins=30,
                    alpha=0.6, color=color, label="Revoked" if label else "Active", density=True)
axes[0, 1].set_title("Years Since Last Filing")
axes[0, 1].legend()

# Asset code distribution
for label, color in [(0, "#4CAF50"), (1, "#F44336")]:
    mask = y == label
    vals = X.loc[mask, "asset_code_usd"]
    vals_log = np.log1p(vals)
    axes[1, 0].hist(vals_log, bins=30, alpha=0.6, color=color,
                    label="Revoked" if label else "Active", density=True)
axes[1, 0].set_title("log(Asset Code USD)")
axes[1, 0].legend()

# Revenue distribution (log scale)
for label, color in [(0, "#4CAF50"), (1, "#F44336")]:
    mask = y == label
    vals = np.log1p(X.loc[mask, "revenue_amount"].clip(lower=0))
    axes[1, 1].hist(vals, bins=40, alpha=0.6, color=color,
                    label="Revoked" if label else "Active", density=True)
axes[1, 1].set_title("log(Revenue)")
axes[1, 1].legend()

plt.suptitle("Feature Distributions by Risk Status", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

<Figure size 1200x800 with 4 Axes>

Clear patterns:
- Revoked orgs tend to be **newer** (younger ruling date)
- Revoked orgs have **stale filings** (years_since_filing is much higher)
- Revoked orgs skew towards **lower assets and revenue**

The filing recency looks like the strongest signal, which makes sense - IRS revokes exempt status for failing to file Form 990 for 3 consecutive years.

## NTEE Sector Analysis

In [6]:
NTEE_LABELS = {
    "A": "Arts", "B": "Education", "C": "Environment", "D": "Animals",
    "E": "Health", "F": "Mental Health", "G": "Disease/Disorders",
    "H": "Medical Research", "I": "Crime/Legal", "J": "Employment",
    "K": "Food/Agriculture", "L": "Housing", "M": "Public Safety",
    "N": "Recreation/Sports", "O": "Youth Development",
    "P": "Human Services", "Q": "International", "R": "Civil Rights",
    "S": "Community Improve.", "T": "Philanthropy",
    "U": "Science/Technology", "V": "Social Science",
    "W": "Public/Society", "X": "Religion",
    "Y": "Mutual Benefit", "Z": "Unknown",
}

df_analysis = X.copy()
df_analysis["revoked"] = y

sector_stats = df_analysis.groupby("ntee_major").agg(
    revocation_rate=("revoked", "mean"),
    count=("revoked", "count")
).sort_values("revocation_rate", ascending=False)

print("Revocation rate by NTEE major sector (top 10):\n")
for code, row in sector_stats.head(10).iterrows():
    label = NTEE_LABELS.get(code, code)
    print(f"{code} ({label:20s}) {row['revocation_rate']*100:5.2f}%  (n={row['count']:,})")

Revocation rate by NTEE major sector (top 10):

Z (Unknown)              8.41%  (n=72,341)
Y (Mutual Benefit)       5.23%  (n=18,294)
N (Recreation/Sports)    4.17%  (n=98,421)
S (Community Improve.)   3.94%  (n=67,832)
O (Youth Development)    3.71%  (n=43,219)
P (Human Services)       3.42%  (n=231,847)
W (Public/Society)       3.18%  (n=52,193)
X (Religion)             2.87%  (n=294,012)
A (Arts)                 2.54%  (n=112,784)
B (Education)            2.31%  (n=187,932)


In [7]:
fig, ax = plt.subplots(figsize=(12, 5))
sector_sorted = sector_stats.sort_values("revocation_rate", ascending=True)
colors = ["#F44336" if r > 0.04 else "#FF9800" if r > 0.03 else "#4CAF50"
          for r in sector_sorted["revocation_rate"]]
ax.barh(range(len(sector_sorted)),
        sector_sorted["revocation_rate"] * 100,
        color=colors)
ax.set_yticks(range(len(sector_sorted)))
ax.set_yticklabels([f"{c} ({NTEE_LABELS.get(c, c)})" for c in sector_sorted.index])
ax.set_xlabel("Revocation Rate (%)")
ax.set_title("Revocation Rate by NTEE Sector")
ax.axvline(x=y.mean()*100, color="gray", linestyle="--", alpha=0.7, label=f"Overall: {y.mean()*100:.2f}%")
ax.legend()
plt.tight_layout()
plt.show()

<Figure size 1200x500 with 1 Axes>

Orgs with missing NTEE codes (Z) have the highest revocation rate at 8.4%. Makes sense - if they didn't classify their mission, they're probably not well-organized. Mutual benefit orgs and recreation/sports also elevated.

## Geographic Distribution

In [8]:
state_stats = df_analysis.groupby("state").agg(
    revocation_rate=("revoked", "mean"),
    count=("revoked", "count")
)
state_stats = state_stats[state_stats["count"] >= 5000].sort_values("revocation_rate", ascending=False)

print("States with highest revocation rates (min 5000 orgs):\n")
for state, row in state_stats.head(10).iterrows():
    print(f"{state}    {row['revocation_rate']*100:.2f}%  (n={row['count']:,})")

States with highest revocation rates (min 5000 orgs):

DC    4.82%  (n=8,421)
NV    4.31%  (n=12,893)
FL    3.87%  (n=89,241)
GA    3.74%  (n=42,187)
TX    3.52%  (n=112,934)
AZ    3.48%  (n=28,421)
IL    3.31%  (n=67,293)
CA    3.24%  (n=164,829)
NY    2.91%  (n=123,491)
OH    2.67%  (n=54,823)


## Feature Correlations

In [9]:
corr_cols = ["asset_code_usd", "income_code_usd", "revenue_amount",
             "subsection_code", "foundation_code", "years_since_ruling",
             "years_since_filing", "filing_req_code", "deductibility_code"]

df_corr = df_analysis[corr_cols + ["revoked"]].copy()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df_corr.corr(), annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

<Figure size 800x600 with 2 Axes>

In [10]:
correlations = df_corr.corr()["revoked"].drop("revoked").sort_values(ascending=False)

print("Point-biserial correlation with revoked label:\n")
notes = {
    "years_since_filing": "(stale filers get revoked)",
    "years_since_ruling": "(newer orgs get revoked more)",
    "income_code_usd": "(lower income = higher risk)",
    "asset_code_usd": "(lower assets = higher risk)",
}
for feat, corr in correlations.items():
    note = notes.get(feat, "")
    print(f"{feat:25s} {corr:7.4f}  {note}")

Point-biserial correlation with revoked label:

years_since_filing     0.3142  (stale filers get revoked)
deductibility_code     0.0834
foundation_code        0.0621
filing_req_code        0.0418
subsection_code        0.0203
years_since_ruling    -0.0847  (newer orgs get revoked more)
revenue_amount        -0.0312
income_code_usd       -0.1204  (lower income = higher risk)
asset_code_usd        -0.1387  (lower assets = higher risk)


Filing recency dominates (0.31 correlation). Asset/income size negatively correlated - smaller orgs are riskier. These align with what the IRS actually uses for auto-revocation (3 years of non-filing).

## SHAP Feature Importance (from trained model)

In [11]:
import json

metadata_path = Path("../models/metadata.json")
with open(metadata_path) as f:
    metadata = json.load(f)

shap_imp = metadata["shap_importance"]

print("Top features by mean |SHAP|:\n")
for feat, val in shap_imp.items():
    print(f"{feat:30s} {val:.6f}")

Top features by mean |SHAP|:

years_since_filing          0.482103
asset_code_usd              0.193847
income_code_usd             0.156291
years_since_ruling          0.098432
foundation_code             0.074218
ntee_major                  0.053891
revenue_amount              0.041273
deductibility_code          0.032847
state                       0.028941
filing_req_code             0.021384
subsection_code             0.014293


In [12]:
features = list(shap_imp.keys())
values = list(shap_imp.values())

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(features)), values[::-1], color="#2196F3")
ax.set_yticks(range(len(features)))
ax.set_yticklabels(features[::-1])
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("Feature Importance (SHAP)")

# Highlight top feature
bars[-1].set_color("#F44336")

plt.tight_layout()
plt.show()

<Figure size 1000x500 with 1 Axes>

`years_since_filing` is the dominant feature by a large margin - almost 2.5x more important than the next feature. This is the IRS auto-revocation mechanism showing up directly in the model.

## Model Performance Summary

In [13]:
metrics = metadata["cv_metrics"]

print("5-Fold Cross-Validation Results:\n")
print(f"  ROC-AUC:           {metrics['roc_auc']}")
print(f"  PR-AUC:            {metrics['pr_auc']}")
print(f"  Precision (rev):   {metrics['precision_revoked']}")
print(f"  Recall (rev):      {metrics['recall_revoked']}")
print(f"  F1 (rev):          {metrics['f1_revoked']}")
print(f"\nTraining details:")
print(f"  Samples:           {metadata['n_train']:,}")
print(f"  Features:          {metadata['n_features']}")
print(f"  Best iteration:    {metadata['xgb_best_iteration']}")
print(f"  scale_pos_weight:  {metadata['scale_pos_weight']}")

5-Fold Cross-Validation Results:

  ROC-AUC:           0.9412
  PR-AUC:            0.6238
  Precision (rev):   0.7124
  Recall (rev):      0.5847
  F1 (rev):          0.6423

Training details:
  Samples:           1,821,403
  Features:          11
  Best iteration:    347
  scale_pos_weight:  32.46


0.94 ROC-AUC is good but PR-AUC at 0.62 reflects the challenge of the class imbalance. The model catches about 58% of actual revocations with 71% precision - reasonable for a screening tool that would flag orgs for human review.

Could potentially improve recall by:
- Lowering the classification threshold (trading precision for recall)
- Adding more features (filing history time series, financial ratios)
- Trying SMOTE or other oversampling on the minority class

In [14]:
# Threshold analysis
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
# Approximate metrics at different thresholds (from eval module output)
precision_vals = [0.42, 0.58, 0.71, 0.81, 0.89, 0.94]
recall_vals = [0.82, 0.72, 0.58, 0.43, 0.28, 0.14]
f1_vals = [2*p*r/(p+r) if (p+r) > 0 else 0 for p, r in zip(precision_vals, recall_vals)]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, precision_vals, "o-", label="Precision", color="#2196F3")
ax.plot(thresholds, recall_vals, "s-", label="Recall", color="#F44336")
ax.plot(thresholds, f1_vals, "^-", label="F1", color="#4CAF50")
ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.5, label="Default threshold")
ax.set_xlabel("Classification Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision-Recall Tradeoff at Different Thresholds")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

<Figure size 800x500 with 1 Axes>

For a fraud screening use case, a lower threshold (~0.3-0.4) might be better - you'd rather flag more orgs for review than miss actual fraud. The default 0.5 is a decent middle ground.